In [2]:
# !pip install requests beautifulsoup4 lxml pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import time
import re
import os
import json

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
BASE    = 'https://en.wikipedia.org'

In [3]:
URL  = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'
resp = requests.get(URL, headers=HEADERS, timeout=15)
assert resp.status_code == 200

soup  = BeautifulSoup(resp.text, 'lxml')
table = soup.find('table', class_='wikitable')

cols = [th.get_text(strip=True) for th in table.find('tr').find_all(['th','td'])]

In [4]:
# Adjust if column order differs from what's printed above
COL_RANK, COL_TITLE, COL_BOX_OFFICE, COL_YEAR = 0, 2, 3, 4

def cell_text(cells, idx):
    if idx >= len(cells): return None
    t = cells[idx].get_text(separator=' ', strip=True)
    return re.sub(r'\[[^\]]*\]', '', t).strip() or None
#get row data from the wikipedia site, take info from the main table
rows_data = []
for row in table.find_all('tr')[1:]:
    cells = row.find_all(['td','th'])
    if len(cells) < 4: continue
    wiki_url = wiki_slug = None
    if COL_TITLE < len(cells): 
        for a in cells[COL_TITLE].find_all('a', href=True):
            if a['href'].startswith('/wiki/'):
                wiki_url  = BASE + a['href']
                wiki_slug = a['href'].replace('/wiki/', '')
                break

    rows_data.append({
        'rank':       cell_text(cells, COL_RANK),
        'title':      cell_text(cells, COL_TITLE),
        'box_office': cell_text(cells, COL_BOX_OFFICE),
        'year':       cell_text(cells, COL_YEAR),
        'wiki_url':   wiki_url,
        'wiki_slug':  wiki_slug,
    })

df = pd.DataFrame(rows_data)
df.head()

,rank,title,box_office,year,wiki_url,wiki_slug
0,1,Avatar,"$2,923,710,708",2009,https://en.wikipedia.org/wiki/Avatar_(2009_film),Avatar_(2009_film)
1,2,Avengers: Endgame,"$2,797,501,328",2019,https://en.wikipedia.org/wiki/Avengers:_Endgame,Avengers:_Endgame
2,3,Avatar: The Way of Water,"$2,334,484,620",2022,https://en.wikipedia.org/wiki/Avatar:_The_Way_...,Avatar:_The_Way_of_Water
3,4,Titanic,"T $2,257,906,828",1997,https://en.wikipedia.org/wiki/Titanic_(1997_film),Titanic_(1997_film)
4,5,Ne Zha 2,"NZ $2,215,690,000",2025,https://en.wikipedia.org/wiki/Ne_Zha_2,Ne_Zha_2


In [30]:
import re
import requests
import time
from bs4 import BeautifulSoup


HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_infobox_field(td):
    """Clean the text of the cell."""
    for tag in td.find_all(['sup', 'style']):
        tag.decompose()
    for span in td.find_all('span', style=lambda v: v and 'display:none' in v.replace(' ', '')):
        span.decompose()

    for br in td.find_all(['br', 'hr']):
        br.replace_with('\n')
    for li in td.find_all('li'):
        li.insert_after('\n')

    raw_text = td.get_text(separator='\n', strip=True)
    
    seen = set()
    out = []
    for line in raw_text.split('\n'):
        line = re.sub(r'\[[^\]]*\]', '', line)  
        line = line.replace('\xa0', ' ').replace('\u200b', '').strip()
        line = re.sub(r'\s+', ' ', line)
        
        if line and line not in seen:
            seen.add(line)
            out.append(line)

    # if there are nothing, return Unknown
    return ', '.join(out) if out else "Unknown"


def scrape_infobox(url):
    # If URL is empty
    if not url or str(url).lower() == 'nan':
        return "Unknown", "Unknown"
        
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code != 200:
            print(f"[{r.status_code}] not able to access {url}")
            return "Unknown", "Unknown"
            
        page = BeautifulSoup(r.text, 'lxml')
        
        # search for infobox
        infoboxes = page.find_all('table', class_=re.compile(r'infobox', re.I))
        
        director = "Unknown"
        country = "Unknown"
        
        for infobox in infoboxes:
            for tr in infobox.find_all('tr'):
                th = tr.find(['th', 'td'], class_=re.compile(r'infobox-label', re.I)) or tr.find('th')
                td = tr.find('td', class_=re.compile(r'infobox-data', re.I)) or tr.find('td')
                
                if not th or not td:
                    continue
                
                label = th.get_text(separator=' ', strip=True).lower().replace('\xa0', ' ')
                
                # catch direct, directed, director, directors, filmmaker
                if ('direct' in label or 'filmmaker' in label) and director == "Unknown":
                    director = get_infobox_field(td)
                    
                # catch country, countries, origin, nationality
                if ('countr' in label or 'origin' in label or 'nation' in label) and country == "Unknown":
                    country = get_infobox_field(td)
            
            # if all of them are not empty, stop searching
            if director != "Unknown" and country != "Unknown":
                break
                
        return director, country
    except Exception as e:
        print(f'{url}: {e}')
        return "Unknown", "Unknown"


In [45]:
# create empty list
directors = []
countries = []

for i, row in df.iterrows():
    d, c = scrape_infobox(row['wiki_url'])
    directors.append(d)
    countries.append(c)
    
    # wait for 1 sec 
    time.sleep(1)

# Write result  to df
df['director'] = directors
df['country'] = countries


[403] not able to access https://en.wikipedia.org/wiki/Spider-Man:_Far_From_Home
[403] not able to access https://en.wikipedia.org/wiki/Transformers:_Dark_of_the_Moon


# Clean all columns 

In [52]:

df_clean = df.drop(columns=['wiki_url', 'wiki_slug']).copy()

# rank & year → integer
df_clean['rank'] = pd.to_numeric(df_clean['rank'], errors='coerce').astype('Int64')
df_clean['year'] = pd.to_numeric(df_clean['year'], errors='coerce').astype('Int64')

# title: remove dagger † (Wikipedia marks for estimated figures)
df_clean['title'] = df_clean['title'].str.replace(r'\s*†\s*', '', regex=True).str.strip()

# box_office: strip prefix abbreviations like 'T $', 'NZ $', 'F8 $'
# Strategy: find the first '$' and parse digits+commas after it
def parse_box_office(val):
    if pd.isna(val): return None
    m = re.search(r'\$([\d,]+)', str(val))
    return float(m.group(1).replace(',', '')) if m else None

df_clean['box_office'] = df_clean['box_office'].apply(parse_box_office)

# country & director: remove empty comma fragments, deduplicate
def clean_list_field(val):
    if pd.isna(val) or val is None: return None
    parts = [p.strip() for p in str(val).split(',')]
    seen, unique = set(), []
    for p in parts:
        if p and p not in seen:
            seen.add(p)
            unique.append(p)
    return ', '.join(unique) or None

df_clean['country']  = df_clean['country'].apply(clean_list_field)
df_clean['director'] = df_clean['director'].apply(clean_list_field)

df_clean = df_clean[['rank', 'title', 'year', 'director', 'box_office', 'country']]


inserting missing values

In [53]:
df.loc[34, 'director'] = 'Jon Watts'
df.loc[34,'country'] = 'United States'

df.loc[36, 'director'] = 'Michael Bay'
df.loc[36,'country'] = 'United States'


In [54]:
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_rows', 60)
display(df_clean)

,rank,title,year,director,box_office,country
0,1,Avatar,2009,James Cameron,2.923711e+09,"United States, United Kingdom"
1,2,Avengers: Endgame,2019,"Anthony Russo, Joe Russo",2.797501e+09,United States
2,3,Avatar: The Way of Water,2022,James Cameron,2.334485e+09,United States
3,4,Titanic,1997,James Cameron,2.257907e+09,United States
4,5,Ne Zha 2,2025,Jiaozi,2.215690e+09,China
5,6,Star Wars: The Force Awakens,2015,J. J. Abrams,2.068224e+09,United States
6,7,Avengers: Infinity War,2018,"Anthony Russo, Joe Russo",2.048360e+09,United States
7,8,Spider-Man: No Way Home,2021,Jon Watts,1.922599e+09,United States
8,9,Zootopia 2,2025,"Jared Bush, Byron Howard",1.866630e+09,United States
9,10,Inside Out 2,2024,Pete Docter,1.698864e+09,United States


# Insert data into database
I used pgAdmin to do it. 

In [12]:
%pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from sqlalchemy import create_engine, text 

# 1. Connetion
db_user = 'postgres'
db_password = ''
db_host = 'localhost'
db_port = '5432'
db_name = 'films'

connection_string = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)

# 2.# 2. Download dataframe to PostgreAdmin
try:
    with engine.begin() as connection:
        connection.execute(text("TRUNCATE TABLE films;"))
        df_clean.to_sql('films', connection, if_exists='append', index=False)
except Exception as e:
    print(f"Error {e}")

## 5. Export to JSON 

In [29]:
df_clean.to_json('films.json', orient='records', force_ascii=False, indent=2)
print('Exported → films.json')

with open('films.json') as f:
    data = json.load(f)
print(json.dumps(data[0], indent=2, ensure_ascii=False))

Exported → films.json
{
  "rank": 1,
  "title": "Avatar",
  "year": 2009,
  "director": "James Cameron",
  "box_office": 2923710708.0,
  "country": "United States, United Kingdom"
}
